## Parallel Design Pattern

#### Import libraries

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

load_dotenv(override=True)

#### Define Model

In [ ]:
model = "gpt-4.1-nano"

# Alternatively, you can use a local model
# client = AsyncOpenAI(base_url="http://localhost:11434/v1")
# model = OpenAIChatCompletionsModel(model = "gpt-oss",openai_client= client)


#### Define Agents

In [ ]:
# Agent 1: GC Content Calculator
gc_agent = Agent(
    name="GCContentAgent",
    instructions="Given a DNA sequence, calculate the GC content percentage. Restrict your output to 50 words or less",
    model=model
)

# Agent 2: Coding Region Predictor
coding_agent = Agent(
    name="CodingRegionAgent",
    instructions="Given a DNA sequence, predict possible coding regions (start and end positions). Restrict your output to 50 words or less",
    model=model
)

# Agent 3: Restriction Site Finder
restriction_agent = Agent(
    name="RestrictionSiteAgent",
    instructions="Given a DNA sequence, list all EcoRI restriction sites (GAATTC) with their positions. Restrict your output to 50 words or less",
    model=model
)


#### Define workflow

In [ ]:
async def parallel_pipeline(dna_sequence):
    # Run all agents concurrently
    results = await asyncio.gather(
        Runner.run(gc_agent, dna_sequence),
        Runner.run(coding_agent, dna_sequence),
        Runner.run(restriction_agent, dna_sequence)
    )
    gc_content = results[0].final_output
    coding_regions = results[1].final_output
    restriction_sites = results[2].final_output

    # Combine results
    report = (
        f"GC Content: {gc_content}\n"
        f"Coding Regions: {coding_regions}\n"
        f"Restriction Sites: {restriction_sites}"
    )
    return report


#### Execute workflow

In [ ]:
# Example usage
from agents import trace
with trace("parallel_pipeline_gpt"):
    dna_sequence = "ATGCGGAATTCGCGTAAATGAATTCGCGT"
    report = await parallel_pipeline(dna_sequence)
    print(report)